# Backtest IC — Analyse des signaux

Ce notebook analyse la capacité prédictive de chaque feature via :
- **IC** : corrélation de rang entre le signal aujourd'hui et le rendement futur
- **ICIR** : régularité du signal dans le temps
- **Decay curve** : à quel horizon le signal est-il le plus fort ?
- **Régime split** : le signal marche-t-il en haute et basse volatilité ?

Un ICIR > 0.10 est généralement considéré comme exploitable en production.

In [1]:
import pathlib, sys
ROOT = pathlib.Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from agents.backtest_ic.ic_engine import orthogonalize, split_regimes, ic_by_regime

RAW      = ROOT / 'data' / 'raw'
EXCHANGE = 'binance-futures'
ASSETS   = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'LTCUSDT', 'BNBUSDT']
HORIZONS = [1, 3, 5, 7]

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.spines.top': False,
                     'axes.spines.right': False})
print('ROOT:', ROOT)

ROOT: /Users/ayman/Downloads/dfi-quant


In [2]:
# ── Chargement des données ────────────────────────────────────────────────────
def load_ohlcv(symbol):
    base = RAW / f'exchange={EXCHANGE}' / 'data_type=ohlcv_1d' / f'symbol={symbol}'
    parts = sorted(base.rglob('part-*.parquet'))
    if not parts:
        return pd.DataFrame()
    df = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
    df['date'] = pd.to_datetime(df['ts_open'], unit='us', utc=True).dt.normalize()
    df = df.sort_values('date').set_index('date')
    df['log_ret'] = np.log(df['close'].astype(float) / df['close'].astype(float).shift(1))
    return df

prices = {sym: load_ohlcv(sym) for sym in ASSETS}
for sym, df in prices.items():
    print(f'{sym}: {len(df)} jours  ({df.index[0].date()} → {df.index[-1].date()})')

BTCUSDT: 2322 jours  (2020-01-01 → 2026-05-10)
ETHUSDT: 2322 jours  (2020-01-01 → 2026-05-10)
SOLUSDT: 2065 jours  (2020-09-14 → 2026-05-10)
LTCUSDT: 2314 jours  (2020-01-09 → 2026-05-10)
BNBUSDT: 2282 jours  (2020-02-10 → 2026-05-10)


In [3]:
# ── Calcul des features ───────────────────────────────────────────────────────
import importlib

FEATURES = {
    'ret_1d':  ('dfi_features.ret_1d',  {}),
    'mom_20d': ('dfi_features.mom_20d', {'window_d': 20}),
    'rv_30d':  ('dfi_features.rv_30d',  {'window_d': 30}),
    'cvd_20d': ('dfi_features.cvd_20d', {'window_d': 20}),
}

signals = {}   # {feature_id: {asset: pd.Series}}
for fid, (module, params) in FEATURES.items():
    mod = importlib.import_module(module)
    signals[fid] = {}
    for sym, df in prices.items():
        s = mod.compute(df, params)
        if not s.empty:
            signals[fid][sym] = s

print('Features calculées:', list(signals.keys()))

Features calculées: ['ret_1d', 'mom_20d', 'rv_30d', 'cvd_20d']


In [ ]:
# ── Calcul IC cross-asset par horizon ─────────────────────────────────────────
def compute_ic_ts(feature_signals, prices, horizons):
    """Pour chaque horizon h, calcule l'IC journalier cross-asset."""
    all_dates = sorted(set.union(*[set(s.index) for s in feature_signals.values()]))
    result = {}
    for h in horizons:
        daily = []
        for t in pd.DatetimeIndex(all_dates):
            xs, xf = [], []
            for sym, sig in feature_signals.items():
                df = prices[sym]
                if t not in sig.index or t not in df.index:
                    continue
                sv = sig.loc[t]
                if pd.isna(sv):
                    continue
                loc = df.index.get_loc(t)
                if loc + h >= len(df):
                    continue
                fwd = float(df['log_ret'].iloc[loc+1:loc+h+1].sum())
                if not pd.isna(fwd):
                    xs.append(sv)
                    xf.append(fwd)
            if len(xs) >= 3:
                ic = pd.Series(xs).corr(pd.Series(xf), method='spearman')
            else:
                ic = float('nan')
            daily.append((t, ic))
        dates, vals = zip(*daily)
        result[h] = pd.Series(vals, index=pd.DatetimeIndex(dates), name=h)
    return result

ic_data = {}
for fid, fsignals in signals.items():
    print(f'Calcul IC pour {fid}...')
    ic_data[fid] = compute_ic_ts(fsignals, prices, HORIZONS)
print('Fait.')

Calcul IC pour ret_1d...
Calcul IC pour mom_20d...


## 1. Tableau récapitulatif — IC moyen et ICIR par feature et horizon

In [ ]:
rows = []
for fid, ic_ts in ic_data.items():
    for h in HORIZONS:
        s = ic_ts[h].dropna()
        icir = s.mean() / s.std() if s.std() > 0 else float('nan')
        rows.append({
            'feature': fid,
            'horizon': f'{h}d',
            'IC moyen': round(s.mean(), 4),
            'ICIR':     round(icir, 3),
            'n':        s.notna().sum(),
        })

summary = pd.DataFrame(rows).pivot_table(
    index='feature', columns='horizon', values=['IC moyen', 'ICIR']
)
print('IC moyen par feature et horizon:')
print(summary['IC moyen'].to_string())
print()
print('ICIR par feature et horizon (|ICIR| > 0.10 = signal exploitable):')
print(summary['ICIR'].to_string())

## 2. Decay curves — IC moyen par horizon

In [ ]:
fig, axes = plt.subplots(1, len(signals), figsize=(14, 4), sharey=False)
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

for ax, (fid, ic_ts), col in zip(axes, ic_data.items(), colors):
    means = [ic_ts[h].mean() for h in HORIZONS]
    stderrs = [ic_ts[h].std() / np.sqrt(ic_ts[h].notna().sum()) for h in HORIZONS]
    bar_colors = ['tab:green' if m >= 0 else 'tab:red' for m in means]
    ax.bar([f'{h}d' for h in HORIZONS], means, color=bar_colors, alpha=0.8,
           yerr=stderrs, capsize=4, ecolor='gray')
    ax.axhline(0, color='k', lw=0.7)
    ax.set_title(fid, fontsize=9)
    ax.set_xlabel('Horizon')
    if ax == axes[0]:
        ax.set_ylabel('IC moyen')

fig.suptitle('Decay curves — IC moyen par horizon (barres = ±1 erreur std)', fontsize=10)
plt.tight_layout()
plt.show()

## 3. IC rolling dans le temps (horizon 1j)

In [ ]:
fig, axes = plt.subplots(len(signals), 1, figsize=(13, 10), sharex=True)

for ax, (fid, ic_ts) in zip(axes, ic_data.items()):
    ic1 = ic_ts[1].dropna()
    roll = ic1.rolling(63, min_periods=20).mean()
    ax.plot(ic1.index, ic1.values, color='lightsteelblue', lw=0.5, alpha=0.5)
    ax.plot(roll.index, roll.values, color='steelblue', lw=1.5, label='rolling 63j')
    ax.axhline(0, color='k', lw=0.6)
    ax.fill_between(roll.index, 0, roll.values,
                    where=roll.values >= 0, color='tab:green', alpha=0.12)
    ax.fill_between(roll.index, 0, roll.values,
                    where=roll.values < 0,  color='tab:red',   alpha=0.12)
    mean_ic = ic1.mean()
    icir = mean_ic / ic1.std()
    ax.set_ylabel(f'{fid}\nIC={mean_ic:.3f} ICIR={icir:.3f}', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))

fig.suptitle('IC rolling 63j — horizon 1 jour (vert = signal positif, rouge = signal négatif)', fontsize=10)
plt.gcf().autofmt_xdate(rotation=30)
plt.tight_layout()
plt.show()

## 4. Régime split — le signal marche-t-il en haute et basse volatilité ?

In [ ]:
btc_rets = prices['BTCUSDT']['log_ret']
regime_labels = split_regimes(btc_rets)

print('Jours par régime:')
print(regime_labels.value_counts().to_string())

fig, axes = plt.subplots(1, len(signals), figsize=(14, 4))

for ax, (fid, ic_ts) in zip(axes, ic_data.items()):
    regime_means = {'high_vol': [], 'low_vol': []}
    for h in HORIZONS:
        aligned = ic_ts[h].reindex(regime_labels.index)
        for reg in ['high_vol', 'low_vol']:
            mask = regime_labels == reg
            regime_means[reg].append(aligned[mask].mean())

    x = np.arange(len(HORIZONS))
    w = 0.35
    ax.bar(x - w/2, regime_means['high_vol'], w, label='Haute vol', color='tab:orange', alpha=0.8)
    ax.bar(x + w/2, regime_means['low_vol'],  w, label='Basse vol',  color='tab:blue',   alpha=0.8)
    ax.axhline(0, color='k', lw=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{h}d' for h in HORIZONS])
    ax.set_title(fid, fontsize=9)
    ax.set_xlabel('Horizon')
    if ax == axes[0]:
        ax.set_ylabel('IC moyen')
        ax.legend(fontsize=7)

fig.suptitle('IC par régime de volatilité BTC — orange=haute vol, bleu=basse vol', fontsize=10)
plt.tight_layout()
plt.show()

## 5. Orthogonalisation — apport unique de chaque feature

On retire l'influence de `mom_20d` et `rv_30d` de chaque feature avant de mesurer l'IC.  
Si l'IC reste similaire → la feature apporte une information **unique**.  
Si l'IC s'effondre → la feature ne fait que redire ce que mom/rv disent déjà.

In [ ]:
ORTHO_CONTROLS = ['mom_20d', 'rv_30d']
H_ORTHO = 1  # horizon 1j pour comparer

rows_ortho = []
for fid, fsignals in signals.items():
    # IC brut
    raw_ic = ic_data[fid][H_ORTHO].mean()

    # IC après orthogonalisation
    ortho_signals = {}
    for sym, sig in fsignals.items():
        controls = pd.DataFrame({
            ctrl: signals[ctrl][sym]
            for ctrl in ORTHO_CONTROLS
            if ctrl != fid and sym in signals.get(ctrl, {})
        })
        if controls.empty:
            ortho_signals[sym] = sig
        else:
            ortho_signals[sym] = orthogonalize(sig, controls)

    ortho_ic_ts = compute_ic_ts(ortho_signals, prices, [H_ORTHO])
    ortho_ic = ortho_ic_ts[H_ORTHO].mean()

    rows_ortho.append({
        'feature': fid,
        'IC brut':  round(raw_ic, 4),
        'IC ortho': round(ortho_ic, 4),
        'delta':    round(ortho_ic - raw_ic, 4),
    })
    print(f'{fid:15s}  IC brut={raw_ic:+.4f}  IC ortho={ortho_ic:+.4f}  delta={ortho_ic-raw_ic:+.4f}')

df_ortho = pd.DataFrame(rows_ortho).set_index('feature')

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(df_ortho))
w = 0.35
ax.bar(x - w/2, df_ortho['IC brut'],  w, label='IC brut',  color='tab:blue',  alpha=0.8)
ax.bar(x + w/2, df_ortho['IC ortho'], w, label='IC ortho', color='tab:orange', alpha=0.8)
ax.axhline(0, color='k', lw=0.6)
ax.set_xticks(x)
ax.set_xticklabels(df_ortho.index)
ax.set_ylabel('IC moyen (horizon 1j)')
ax.set_title('IC brut vs IC après orthogonalisation (mom_20d + rv_30d retirés)')
ax.legend()
plt.tight_layout()
plt.show()